<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.2-embeddings/notebooks/GCP_Capstone_2.2_Embeddings.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2.2 Embeddings — Text to Vectors
**Netsetos GenAI Engineering — GCP Capstone**

Master embedding APIs, cosine similarity, task types, cross-language search, and build a production embeddings module.


## Setup


In [ ]:
!pip install -q google-genai numpy scikit-learn matplotlib
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
import numpy as np

client = genai.Client(vertexai=True, project=PROJECT_ID, location='us-central1')


## Cell 1: First Embedding


In [ ]:
response = client.models.embed_content(
    model='gemini-embedding-001',
    contents='Machine learning is a subset of artificial intelligence.',
    config=types.EmbedContentConfig(
        task_type='RETRIEVAL_DOCUMENT', output_dimensionality=768),
)
vec = response.embeddings[0].values
print(f'Dimensions: {len(vec)}')
print(f'First 5: {[round(v,4) for v in vec[:5]]}')
print(f'Tokens: {response.embeddings[0].statistics.token_count}')


## Cell 2: Cosine Similarity


In [ ]:
def cosine_sim(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

texts = [
    'How do I reset my password?',
    'I forgot my login credentials',
    'What is the weather like today?',
    'Is it going to rain tomorrow?',
]

resp = client.models.embed_content(
    model='gemini-embedding-001', contents=texts,
    config={'task_type':'SEMANTIC_SIMILARITY','output_dimensionality':768})
vecs = [e.values for e in resp.embeddings]

for i in range(len(texts)):
    for j in range(i+1, len(texts)):
        sim = cosine_sim(vecs[i], vecs[j])
        print(f'  {sim:.4f} | {texts[i][:30]} <-> {texts[j][:30]}')


## Cell 3: Task Type Impact — Asymmetric vs Symmetric


In [ ]:
doc = 'The aurora borealis occurs when solar wind particles collide with atmospheric gases.'
query = 'What causes the northern lights?'

# Correct: asymmetric
d_r = client.models.embed_content(model='gemini-embedding-001', contents=doc,
    config={'task_type':'RETRIEVAL_DOCUMENT','output_dimensionality':768}).embeddings[0].values
q_r = client.models.embed_content(model='gemini-embedding-001', contents=query,
    config={'task_type':'RETRIEVAL_QUERY','output_dimensionality':768}).embeddings[0].values

# Wrong: symmetric
d_s = client.models.embed_content(model='gemini-embedding-001', contents=doc,
    config={'task_type':'SEMANTIC_SIMILARITY','output_dimensionality':768}).embeddings[0].values
q_s = client.models.embed_content(model='gemini-embedding-001', contents=query,
    config={'task_type':'SEMANTIC_SIMILARITY','output_dimensionality':768}).embeddings[0].values

print(f'Asymmetric (correct): {cosine_sim(d_r, q_r):.4f}')
print(f'Symmetric (wrong):    {cosine_sim(d_s, q_s):.4f}')


## Cell 4: Mini Semantic Search Engine


In [ ]:
corpus = [
    'Python is a popular programming language for data science.',
    'React is a JavaScript library for building UIs.',
    'Gradient descent optimizes neural network weights.',
    'Hyderabad is home to major tech companies.',
    'The transformer architecture uses self-attention.',
    'RAG combines retrieval with language generation.',
]

doc_r = client.models.embed_content(
    model='gemini-embedding-001', contents=corpus,
    config={'task_type':'RETRIEVAL_DOCUMENT','output_dimensionality':768})
doc_vecs = np.array([e.values for e in doc_r.embeddings])
doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

for query in ['How does attention work?', 'Best city for tech jobs in India?']:
    q_r = client.models.embed_content(
        model='gemini-embedding-001', contents=query,
        config={'task_type':'RETRIEVAL_QUERY','output_dimensionality':768})
    q_v = np.array(q_r.embeddings[0].values)
    q_v = q_v / np.linalg.norm(q_v)
    sims = doc_vecs @ q_v
    top3 = np.argsort(sims)[::-1][:3]
    print(f'\nQuery: {query}')
    for r, i in enumerate(top3): print(f'  {r+1}. [{sims[i]:.4f}] {corpus[i]}')


## Cell 5: Cross-Language Similarity


In [ ]:
pairs = [
    ('AI is transforming healthcare', '\u0915\u0943\u0924\u094d\u0930\u093f\u092e \u092c\u0941\u0926\u094d\u0927\u093f\u092e\u0924\u094d\u0924\u093e \u0938\u094d\u0935\u093e\u0938\u094d\u0925\u094d\u092f \u0938\u0947\u0935\u093e \u0915\u094b \u092c\u0926\u0932 \u0930\u0939\u0940 \u0939\u0948', 'healthcare'),
    ('Python is great for ML', '\u092a\u093e\u092f\u0925\u0928 ML \u0915\u0947 \u0932\u093f\u090f \u092c\u0939\u0941\u0924 \u0905\u091a\u094d\u091b\u093e \u0939\u0948', 'python'),
]
all_t = []
for en, hi, _ in pairs: all_t.extend([en, hi])
resp = client.models.embed_content(model='gemini-embedding-001', contents=all_t,
    config={'task_type':'SEMANTIC_SIMILARITY','output_dimensionality':768})
vecs = [e.values for e in resp.embeddings]
for i, (en, hi, label) in enumerate(pairs):
    sim = cosine_sim(vecs[i*2], vecs[i*2+1])
    print(f'  {sim:.4f} | EN-HI: {label}')


## Cell 6: PCA Visualization


In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

categories = {
    'AI/ML': ['Neural networks learn from data', 'Transformers use attention', 'GPT is a language model'],
    'Food': ['Biryani uses basmati rice', 'Dosa is a South Indian crepe', 'Butter chicken is popular'],
    'Sports': ['Cricket is popular in India', 'Football is global', 'Olympics happen every 4 years'],
}
all_texts, labels = [], []
for cat, texts in categories.items():
    all_texts.extend(texts); labels.extend([cat]*len(texts))

resp = client.models.embed_content(model='gemini-embedding-001', contents=all_texts,
    config={'task_type':'CLUSTERING','output_dimensionality':768})
vecs = np.array([e.values for e in resp.embeddings])
coords = PCA(n_components=2).fit_transform(vecs)

for cat in categories:
    mask = [l==cat for l in labels]
    plt.scatter(coords[np.array(mask),0], coords[np.array(mask),1], label=cat, s=120)
plt.legend(); plt.title('Embedding Clusters'); plt.show()


## Cell 7: Production Embeddings Module


In [ ]:
MODEL = 'gemini-embedding-001'
DIMS = 768

def normalize(vec):
    v = np.array(vec)
    return (v / np.linalg.norm(v)).tolist()

def embed_documents(client, texts, batch_size=100):
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        r = client.models.embed_content(
            model=MODEL, contents=texts[i:i+batch_size],
            config={'task_type':'RETRIEVAL_DOCUMENT','output_dimensionality':DIMS})
        all_vecs.extend([normalize(e.values) for e in r.embeddings])
    return all_vecs

def embed_query(client, query):
    r = client.models.embed_content(
        model=MODEL, contents=query,
        config={'task_type':'RETRIEVAL_QUERY','output_dimensionality':DIMS})
    return normalize(r.embeddings[0].values)

def search(q_vec, doc_vecs, texts, top_k=3):
    sims = np.array(doc_vecs) @ np.array(q_vec)
    top = np.argsort(sims)[::-1][:top_k]
    return [(texts[i], float(sims[i])) for i in top]

# Test
docs = ['Python is great for ML', 'Hyderabad has amazing food',
        'Docker simplifies deployment', 'Transformers use attention']
vecs = embed_documents(client, docs)
q = embed_query(client, 'How does deep learning work?')
for text, score in search(q, vecs, docs):
    print(f'  [{score:.4f}] {text}')


## ✅ Lesson 2.2 Complete!

- ✅ Embeddings = vectors where proximity equals meaning
- ✅ Cosine similarity measures semantic closeness
- ✅ gemini-embedding-001 > text-embedding-005 on quality
- ✅ Task types: RETRIEVAL_DOCUMENT + RETRIEVAL_QUERY (asymmetric)
- ✅ Cross-language search works natively
- ✅ 768d = 99.74% quality at 25% storage
- ✅ embeddings.py module built and tested

**Next: Lesson 2.3 — Vector Search with Firestore**
